In [1]:
# Install Libraries

!pip install -q transformers datasets accelerate sentencepiece

In [2]:
# Imports

import os
import json
import random
import torch
import numpy as np

from tqdm import tqdm
from datasets import load_dataset

from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
# Reproducability

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
# Enable TF32

torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

In [5]:
# Mount Drive

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


To generate data for `wiki text`, run the next cell as it is.  
If you want the `khan academy` data, comment the 2nd line and un-comment the 3rd line and run.

In [ ]:
# Layer Selection

TARGET_LAYER = 16

# TARGET_LAYER can take any of the following values: 4, 8, 12, 16, 20, 28, 31.   
# Choose a value and re-run the whole thing from that point or it's better to disconnect and delete the run time and then run.   
# I've done this for 7 times and got the results for each layer.  
# I could have done it using a loop but the free trail has limitatins.

In [ ]:
# Paths

PROJECT_DIR = "/content/drive/MyDrive/Natural Language Autoencoder/"
DATA_DIR = os.path.join(PROJECT_DIR, "data", "wikitext")
# DATA_DIR = os.path.join(PROJECT_DIR, "data", "khanacademy")

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

DATASET_JSON_PATH = os.path.join(DATA_DIR, "dataset.json")
ACTIVATIONS_PATH = os.path.join(DATA_DIR, f"layer_{TARGET_LAYER}.pt")

In [7]:
# Load WikiText

dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")

texts = []

for item in dataset:
    text = item["text"].strip()

    if len(text.split()) > 20:
        texts.append(text)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [ ]:
# Shuffle the data

random.shuffle(texts)

texts = texts[:5000]

print("Total Samples:", len(texts))

Total Samples: 2000


In [9]:
# Load Phi-3

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map="auto")

model.eval()

print("Phi-3 loaded.")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Phi-3 loaded.


In [11]:
# Prompt Template
def build_prompt(text):

    prompt = f"""
You are a semantic labeling engine.

Read the given text and output ONLY a concise semantic topic label.

Rules:
- Maximum 10 words
- No punctuation
- No explanations
- No full sentences
- Output only the semantic label
- Capture the main entities events or topics
- Avoid generic labels

Examples:
Ancient Roman political history
Neural network optimization methods
Egyptian mythology divine kingship conflict
Video game franchise development
Michael Jordan basketball dominance

TEXT:
{text}

LABEL:
"""

    return prompt.strip()

In [12]:
# Containers

dataset_entries = []
activation_list = []

In [13]:
# Main Processing Loop

for text in tqdm(texts):
    try:
        prompt = build_prompt(text)

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(model.device)

        # FORWARD PASS (for activations)
        with torch.no_grad():

            outputs = model(
                **inputs,
                output_hidden_states=True
            )

        hidden_states = outputs.hidden_states

        target_hidden = hidden_states[TARGET_LAYER]

        last_activation = target_hidden[:, -1, :].squeeze(0).cpu()

        # GENERATE DESCRIPTION
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id # Added new
            )

        generated_text = tokenizer.decode(
            generated_ids[0],
            skip_special_tokens=True
        )

        description = generated_text[len(prompt):]
        description = description.strip().split("\n")[0].strip()

        # -----------------------------
        # SAVE
        # -----------------------------

        dataset_entries.append({"text": text, "phi3_description": description})

        activation_list.append(last_activation)

    except Exception as e:
        print("Error:", e)

100%|██████████| 2000/2000 [35:49<00:00,  1.07s/it]


In [14]:
# Stack Activations

activations_tensor = torch.stack(activation_list)

print(activations_tensor.shape)

torch.Size([2000, 3072])


In [15]:
# Save dataset.json

with open(DATASET_JSON_PATH, "w") as f:

    json.dump(dataset_entries, f, indent=2)

print("dataset.json saved.")

dataset.json saved.


In [16]:
# Save activations.pt

torch.save(activations_tensor, ACTIVATIONS_PATH)

print("activations.pt saved.")

activations.pt saved.


In [17]:
# Verify

print(dataset_entries[0])

print(activations_tensor.shape)

{'text': "In the middle of 1864 the Beetons again visited the Goubauds in Paris — the couple 's third visit to the city — and Isabella was pregnant during the visit , just as she had been the previous year . On her return to Britain she began working on an abridged version of the Book of Household Management , which was to be titled The Dictionary of Every @-@ Day Cookery . On 29 January 1865 , while working on the proofs of the dictionary , she went into labour ; the baby — Mayson Moss — was born that day . Isabella began to feel feverish the following day and died of puerperal fever on 6 February at the age of 28 .", 'phi3_description': "Isabella Beeton's life and death"}
torch.Size([2000, 3072])
